# Thành viên 1 — Kaggle `GPU T4 x 2` (32 GB VRAM): Track Tiếng Việt + Mô hình NLP (`Qwen3.8-27B QLoRA -> ViSoBERT`)

**Cấu hình bắt buộc trên Kaggle trước khi chạy:**
1. Bật **Accelerator: GPU T4 x 2** (tổng 32 GB VRAM).
2. Bật **Internet: ON** để kéo repo GitHub, dữ liệu Arctic Shift và trọng số Hugging Face.
3. Bật **Persistence: Files only** (khuyên dùng) để giữ checkpoint nếu ngắt phiên.

Bạn có 2 chế độ chọn nhanh:
- **`member1_vn`** *(Khuyên dùng để chia 50/50 với Bạn 2)*: Chạy trọn gói Tiếng Việt (thu thập 1.500 cây Voz/VietNam -> tinh chỉnh `Qwen3.8-27B QLoRA` -> chưng cất 4D `ViSoBERT` -> vòng lặp tự tối ưu & 5-seed `SBi-FAGAT v2` + xuất file chấm 300 câu).
- **`model_nlp_only`**: Nếu muốn 1 người chuyên train toàn bộ **Mô hình NLP (`Qwen3.8-27B QLoRA + Cross-Encoder 4D`)** xuất sẵn `.pt` rồi gửi sang cho Bạn 2 train GNN.

In [ ]:
# Bước 1: Kiểm tra đủ 2 card NVIDIA Tesla T4 (tổng ~30-32 GB VRAM)
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available(), "| Num GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}:", torch.cuda.get_device_name(i))

In [ ]:
# Bước 2: Clone hoặc cập nhật mã nguồn mới nhất từ GitHub & cài thư viện T4 x 2
%cd /kaggle/working
!if [ ! -d "gnn-nlp-propagation" ]; then git clone https://github.com/hieu-kiien/gnn-nlp-propagation.git; fi
%cd /kaggle/working/gnn-nlp-propagation
!git pull origin main
!pip install -q -U -r requirements-kaggle-t4x2.txt
!pytest -q

In [ ]:
# (Tùy chọn) Đăng nhập Hugging Face nếu repo model yêu cầu token đọc
# from kaggle_secrets import UserSecretsClient
# import os
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
# Bước 3: Chạy thử kiểm tra nhanh hệ thống (chạy trong <10 giây để chắc chắn 0 lỗi môi trường)
!python -m scripts.kaggle_parallel_runner --role member1_vn --smoke-test
!python -m scripts.predict_live_thread --demo-offline

In [ ]:
# Bước 4: CHẠY CHÍNH THỨC TRÊN KAGGLE T4 x 2
# Đổi ROLE = "model_nlp_only" nếu bạn chỉ muốn chuyên trách xuất bộ NLP Tiếng Việt (.pt) cho bạn còn lại
# Dùng bản nén sẵn 4-bit NF4 (22.4 GB) của Qwen/Qwen3.8-27B để vừa khít ổ đĩa Kaggle T4 x 2
TEACHER_MODEL = "unsloth/Qwen3.8-27B-unsloth-bnb-4bit"  # hoặc "Qwen/Qwen3.8-27B"
ROLE = "member1_vn"  # hoặc "model_nlp_only"
VN_TARGET_COUNT = 1500

!python -m scripts.kaggle_parallel_runner \
    --role {ROLE} \
    --lang vi \
    --vn-target-count {VN_TARGET_COUNT} \
    --max-teacher-pairs 6000 \
    --teacher-model {TEACHER_MODEL} \
    --scoring-mode logit \
    --hidden-dim 256 \
    --epochs-nlp 3 \
    --epochs-gnn 100

In [ ]:
# Bước 5: Kiểm tra thử trực tiếp 1 bài Reddit thật trên mạng bằng checkpoint vừa train
# Thay --reddit-url bằng link bài tranh luận mới bất kỳ trên r/vozforums hoặc r/VietNam
!python -m scripts.predict_live_thread \
    --demo-offline \
    --cross-encoder-ckpt checkpoints/dual_ce_vn_v2 \
    --gnn-ckpt outputs/auto_loop_vn/best_validated_sbifagat.pt

## Xuất kết quả gửi lại AI chấm & phân tích lỗi
Sau khi ô trên chạy xong, tải file `sbifagat_member1_vn_bundle.zip` (hoặc `sbifagat_model_nlp_only_bundle.zip`) trong mục `/kaggle/working/gnn-nlp-propagation/`.
File quan trọng nhất để gửi cho AI kiểm tra vòng lặp là:
- `outputs/auto_loop_vn/loop_diagnostics.json`
- `outputs/auto_loop_vn/worst_val_cases_for_review.json`
- `paper/results_main_vn.json` và `data/interim/human_audit_vi_300.csv` (để 2 bạn chấm chéo 300 câu).